# 🦷 Dental AI — YOLOv8 Model Performans Analizi
## V4 vs V5 Karşılaştırması

Bu notebook, diş eti hastalığı tespiti için eğitilen YOLOv8 modelinin (v5) performansını analiz etmektedir.  
**Kullanılan teknikler:** Stratified Dataset Split, Minority Class Oversampling, Augmentation iyileştirmeleri

---

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
from pathlib import Path
from collections import Counter
from ultralytics import YOLO

rcParams['font.family'] = 'DejaVu Sans'
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False

# Sinif isimleri
CLASS_NAMES = {
    0: 'Sağlıklı',
    1: 'Hafif Gingivitis',
    2: 'İleri Gingivitis',
    3: 'Periodontitis',
    4: 'Plak',
    5: 'Tartar',
    6: 'Kanama'
}

COLORS = ['#4CAF50','#2196F3','#FF9800','#F44336','#9C27B0','#00BCD4','#FF5722']

print('Kütüphaneler yüklendi ✓')

## 1. Dataset Dağılım Analizi
Stratified split öncesi ve sonrası sınıf dağılımları

In [ ]:
def get_class_distribution(base_path):
    """Dataset klasöründeki sınıf dağılımını hesaplar."""
    splits = ['training', 'validation', 'test']
    result = {}
    for split in splits:
        counts = Counter()
        lbl_dir = Path(base_path) / split / 'labels'
        if not lbl_dir.exists():
            continue
        for lf in lbl_dir.glob('*.txt'):
            if '_os' in lf.stem:
                continue
            for line in lf.read_text().splitlines():
                if line.strip():
                    counts[int(line.split()[0])] += 1
        result[split] = counts
    return result

# Mevcut dataset (stratified split sonrasi)
dist = get_class_distribution('Dataset')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Dataset Sınıf Dağılımı (Stratified Split Sonrası)', fontsize=14, fontweight='bold', y=1.02)

for ax, (split, counts) in zip(axes, dist.items()):
    names = [CLASS_NAMES[i] for i in range(7)]
    values = [counts.get(i, 0) for i in range(7)]
    bars = ax.bar(range(7), values, color=COLORS, edgecolor='white', linewidth=0.8)
    ax.set_xticks(range(7))
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=9)
    ax.set_title(f'{split.capitalize()} ({sum(values):,} bbox)', fontweight='bold')
    ax.set_ylabel('Bounding Box Sayısı')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('dataset_dagilim.png', dpi=150, bbox_inches='tight')
plt.show()
print('Dataset dağılım grafiği kaydedildi.')

## 2. V4 vs V5 — mAP50 Karşılaştırması
Sınıf bazlı model performans karşılaştırması

In [ ]:
# V4 ve V5 sonuclari (validation uzerinden)
v4_map50 = {
    'Sağlıklı':         0.995,
    'Hafif Gingivitis': 0.995,
    'İleri Gingivitis': 0.129,
    'Periodontitis':    0.174,
    'Plak':             0.462,
    'Tartar':           0.446,
    'Kanama':           0.620,
}

v5_map50 = {
    'Sağlıklı':         0.995,
    'Hafif Gingivitis': 0.995,
    'İleri Gingivitis': 0.990,
    'Periodontitis':    0.921,
    'Plak':             0.860,
    'Tartar':           0.798,
    'Kanama':           0.832,
}

labels = list(v4_map50.keys())
v4_vals = list(v4_map50.values())
v5_vals = list(v5_map50.values())

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width/2, v4_vals, width, label='V4 (Önceki Model)',
               color='#90CAF9', edgecolor='white')
bars2 = ax.bar(x + width/2, v5_vals, width, label='V5 (Yeni Model)',
               color='#1565C0', edgecolor='white')

ax.set_xlabel('Sınıf', fontsize=11)
ax.set_ylabel('mAP@50', fontsize=11)
ax.set_title('V4 vs V5 — Sınıf Bazlı mAP50 Karşılaştırması', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=10)
ax.axhline(y=0.7, color='orange', linestyle='--', alpha=0.6, label='Kabul edilebilir eşik (0.70)')
ax.axhline(y=0.85, color='green', linestyle='--', alpha=0.6, label='İyi eşik (0.85)')

# Degisim oklarini goster
for i, (v4, v5) in enumerate(zip(v4_vals, v5_vals)):
    delta = v5 - v4
    color = '#1B5E20' if delta > 0.1 else '#4CAF50' if delta > 0 else '#F44336'
    ax.text(i, max(v4, v5) + 0.03, f'+{delta:.2f}' if delta >= 0 else f'{delta:.2f}',
            ha='center', fontsize=8, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('v4_v5_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()
print('Karşılaştırma grafiği kaydedildi.')

## 3. Genel Metrikler — Özet Tablo

In [ ]:
# Genel metrikler
metrics = {
    'Metrik':       ['Precision', 'Recall', 'mAP@50', 'mAP@50-95'],
    'V4':           [0.537,       0.651,    0.546,     0.372],
    'V5':           [0.872,       0.879,    0.908,     0.768],
    'İyileşme (%)': [None, None, None, None]
}

for i in range(4):
    delta = ((metrics['V5'][i] - metrics['V4'][i]) / metrics['V4'][i]) * 100
    metrics['İyileşme (%)'][i] = f'+{delta:.1f}%'

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

table_data = []
for i in range(4):
    table_data.append([
        metrics['Metrik'][i],
        f"{metrics['V4'][i]:.3f}",
        f"{metrics['V5'][i]:.3f}",
        metrics['İyileşme (%)'][i]
    ])

table = ax.table(
    cellText=table_data,
    colLabels=['Metrik', 'V4 (Eski)', 'V5 (Yeni)', 'İyileşme'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)

table.auto_set_font_size(False)
table.set_fontsize(11)

# Baslik satiri rengi
for j in range(4):
    table[(0, j)].set_facecolor('#1565C0')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Iyilestirme sutunu yesil
for i in range(1, 5):
    table[(i, 3)].set_facecolor('#E8F5E9')
    table[(i, 3)].set_text_props(color='#1B5E20', fontweight='bold')
    table[(i, 2)].set_facecolor('#E3F2FD')

ax.set_title('Genel Model Performansı — V4 vs V5', fontsize=13, fontweight='bold', pad=20)
plt.savefig('genel_metrikler.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. V5 Model Validation — Gerçek Zamanlı Hesaplama
Modeli yükleyip validation setinde çalıştır

In [ ]:
model_path = 'runs/detect/dental_strong_m_v5/weights/best.pt'

if Path(model_path).exists():
    model = YOLO(model_path)
    metrics = model.val(data='dataset.yaml', split='val', verbose=True)
    
    # Sinif bazli map50
    class_map50 = metrics.box.ap50  # array
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # Sol: sinif bazli map50
    ax = axes[0]
    bars = ax.barh([CLASS_NAMES[i] for i in range(7)], class_map50, color=COLORS)
    ax.axvline(x=0.7, color='orange', linestyle='--', alpha=0.7, label='Kabul eşiği')
    ax.axvline(x=0.85, color='green', linestyle='--', alpha=0.7, label='İyi eşik')
    ax.set_xlabel('mAP@50')
    ax.set_title('V5 — Sınıf Bazlı mAP@50', fontweight='bold')
    ax.legend()
    ax.set_xlim(0, 1.1)
    for bar, val in zip(bars, class_map50):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    
    # Sag: precision-recall
    ax2 = axes[1]
    precision = metrics.box.p
    recall = metrics.box.r
    x = np.arange(len(CLASS_NAMES))
    ax2.bar(x - 0.2, precision, 0.4, label='Precision', color='#42A5F5')
    ax2.bar(x + 0.2, recall, 0.4, label='Recall', color='#66BB6A')
    ax2.set_xticks(x)
    ax2.set_xticklabels([CLASS_NAMES[i] for i in range(7)], rotation=30, ha='right', fontsize=9)
    ax2.set_ylabel('Değer')
    ax2.set_ylim(0, 1.15)
    ax2.set_title('V5 — Precision & Recall (Sınıf Bazlı)', fontweight='bold')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig('v5_detay_metrikler.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\n✅ Genel mAP50: {metrics.box.map50:.3f}')
    print(f'✅ Genel mAP50-95: {metrics.box.map:.3f}')
else:
    print(f'Model bulunamadı: {model_path}')

## 5. Eğitim Süreci — Loss & mAP Eğrileri

In [ ]:
import pandas as pd

results_csv = Path('runs/detect/dental_strong_m_v5/results.csv')

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('V5 Model — Eğitim Süreci', fontsize=14, fontweight='bold')
    
    # Box loss
    ax = axes[0, 0]
    ax.plot(df['epoch'], df['train/box_loss'], label='Train', color='#F44336')
    ax.plot(df['epoch'], df['val/box_loss'], label='Val', color='#FF9800')
    ax.set_title('Box Loss')
    ax.set_xlabel('Epoch')
    ax.legend()
    
    # Cls loss
    ax = axes[0, 1]
    ax.plot(df['epoch'], df['train/cls_loss'], label='Train', color='#9C27B0')
    ax.plot(df['epoch'], df['val/cls_loss'], label='Val', color='#E91E63')
    ax.set_title('Classification Loss')
    ax.set_xlabel('Epoch')
    ax.legend()
    
    # mAP50
    ax = axes[1, 0]
    ax.plot(df['epoch'], df['metrics/mAP50(B)'], color='#1565C0', linewidth=2)
    ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.6)
    ax.set_title('mAP@50')
    ax.set_xlabel('Epoch')
    ax.set_ylim(0, 1.05)
    ax.fill_between(df['epoch'], df['metrics/mAP50(B)'], alpha=0.1, color='#1565C0')
    
    # Precision & Recall
    ax = axes[1, 1]
    ax.plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#42A5F5')
    ax.plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='#66BB6A')
    ax.set_title('Precision & Recall')
    ax.set_xlabel('Epoch')
    ax.set_ylim(0, 1.05)
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('egitim_sureci.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Eğitim süreci grafikleri kaydedildi.')
else:
    print(f'results.csv bulunamadı: {results_csv}')

## 6. Confusion Matrix

In [ ]:
# YOLO'nun otomatik kaydettiği confusion matrix gorselini goster
import matplotlib.image as mpimg

cm_path = Path('runs/detect/dental_strong_m_v5/confusion_matrix_normalized.png')
if not cm_path.exists():
    cm_path = Path('runs/detect/dental_strong_m_v5/confusion_matrix.png')

if cm_path.exists():
    img = mpimg.imread(cm_path)
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Confusion Matrix — V5', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Confusion matrix görseli bulunamadı.')
    print('Şu komutla üretebilirsiniz:')
    print('  model.val(data="dataset.yaml", plots=True)')

## 7. Özet ve Sonuç

### 🎯 Yapılan İyileştirmeler

| Teknik | Açıklama |
|--------|----------|
| **Stratified Split** | Sınıf dağılımı korunarak dataset yeniden bölündü |
| **Minority Oversampling** | Az örnekli sınıflar 4x çoğaltıldı |
| **Augmentation** | Mosaic 0.3→0.8, copy_paste=0.1 eklendi |
| **Loss Ağırlığı** | cls=1.5 ile sınıf kaybı ağırlıklandırıldı |

### 📊 Sonuçlar

| Sınıf | V4 mAP50 | V5 mAP50 | İyileşme |
|-------|----------|----------|----------|
| İleri Gingivitis | 0.129 | 0.990 | **+0.861** |
| Periodontitis | 0.174 | 0.921 | **+0.747** |
| Plak | 0.462 | 0.860 | **+0.398** |
| Tartar | 0.446 | 0.798 | **+0.352** |
| **Genel** | **0.546** | **0.908** | **+0.362** |

### 💡 Önemli Bulgu
V4 modelinde düşük performansın temel nedeni **validation setindeki dengesiz dağılım** idi (İleri Gingivitis: 15 görüntü). Stratified split ile bu sorun çözüldü.